# Demo: Branching MASCAF

Fit a cable morphology to the branching demo mesh. The skeleton stops short of the branch ends, so the fit is extended at each tip and then scaled to the mesh surface area.

In [46]:
from mascaf import *

import logging

logging.basicConfig(level=logging.INFO)

## Mesh and skeleton

Load the demo mesh and its curve skeleton. The skeleton is the centerline the cable fit follows.

In [47]:
mm = MeshManager(mesh_path="../../data/demo/branching.obj")
mm.print_mesh_analysis()
raw_skeleton = SkeletonGraph.from_txt(f"../../data/demo/branching.polylines.txt")
# raw_skeleton.prune_short_branches_inplace(min_length_fraction=1)
mesh_fig, camera = visualize_mesh_3d(
    mesh=mm,
    title="Branching test model",
    skel=raw_skeleton,
    show_axes=False,
    eye_scale=0.9,
    return_camera=True,
)
mesh_fig

INFO:mascaf.mesh:Loaded mesh: 1417 vertices, 2830 faces
INFO:mascaf.mesh:Mesh Analysis Report
INFO:mascaf.mesh:====================
INFO:mascaf.mesh:
Geometry:
INFO:mascaf.mesh:  * Vertices: 1417
INFO:mascaf.mesh:  * Faces: 2830
INFO:mascaf.mesh:  * Components: 1
INFO:mascaf.mesh:  * Volume: 5.11
INFO:mascaf.mesh:  * Bounds: [-1.0, -1.3, -3.6] to [1.0, 1.4, 1.0]
INFO:mascaf.mesh:
Mesh Quality:
INFO:mascaf.mesh:  * Watertight: True
INFO:mascaf.mesh:  * Winding Consistent: True
INFO:mascaf.mesh:  * Normal Direction: outward
INFO:mascaf.mesh:  * Duplicate Vertices: 0
INFO:mascaf.mesh:  * Degenerate Faces: 0
INFO:mascaf.mesh:
Topology:
INFO:mascaf.mesh:  * Genus: 0
INFO:mascaf.mesh:  * Euler Characteristic: 2
INFO:mascaf.mesh:
No issues detected
INFO:mascaf.mesh:
Recommendation:
INFO:mascaf.mesh:  Mesh appears to be in good condition.
INFO:mascaf.mesh:====================


## Cable fit

Basis optimization snaps nodes onto the mesh and pulls them toward the centerline. `CableFitter` then resamples that skeleton so edges are at most `max_edge_length` and estimates a radius at each sample from the mesh cross-section.

In [48]:
max_edge_length = 1.0
optimizer_options = BasisOptimizerOptions(
    # do_pruning=True,
    # pruning_length_fraction=0.2,
    # do_snapping=True,
    # do_forcing=True,
    # max_iterations=20,
    # alpha_s=0.1,
    # step_scale=0.5,
    # n_rays=12,
    # ray_jitter=0.1,
    # localization_beta=2.0,
    # preserve_terminal_nodes=True,
    # preserve_branch_nodes=False,
)

skeleton = raw_skeleton

In [49]:
swc_filepath = "../../data/demo/branching.swc"

morph = CableFitter(
    FitOptions(
        max_edge_length=max_edge_length,
        basis_optimizer_options=optimizer_options,
    )
).fit(
    mm.mesh,
    skeleton,
)

INFO:mascaf.cable_fitting:Starting cable fit with 137 skeleton nodes, 136 skeleton edges, 1417 mesh vertices, and max_edge_length=1.0
INFO:mascaf.morphology_graph:Resampled morphology: 9 nodes, 8 edges (max_edge_length=1.0000, source 137 nodes / 136 edges, 5 sections)
INFO:mascaf.cable_fitting:Optimizing morphology basis before radius fitting
INFO:mascaf.basis_optimizer:Starting basis optimization...
INFO:mascaf.basis_optimizer:  Input: 9 nodes, 8 edges, total length 5.8145
INFO:mascaf.basis_optimizer:Phase 1 - Snapping: 0 nodes outside mesh
INFO:mascaf.basis_optimizer:Basis optimization complete: 9 nodes, 8 edges, 0 outside, total length 5.8145
INFO:mascaf.cable_fitting:Computing node radii: strategy=equivalent_area, 9 nodes, probe_eps=1.00e-04, tries=3
INFO:mascaf.cable_fitting:Radii computed: min=0.161176 max=0.958907 mean=0.390188 median=0.313264
INFO:mascaf.cable_fitting:Finished cable fit with 9 morphology nodes and 8 morphology edges


## Fitted cable

The mesh and skeleton are on the left; the cable model is on the right. Both panels use the same view. Validation compares cable volume and surface area with the mesh.

In [50]:
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=8 sides=16 end_caps=False verts=256 faces=256
INFO:swctools.geometry:FrustaSet.from_swc_model edges=8 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=8 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 9 nodes, 8 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 3.8648
INFO:mascaf.validation:---- Ratio:             0.7567
INFO:mascaf.validation:---- Error:             -1.2424
INFO:mascaf.validation:---- Relative error:    -24.33%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   19.5395
INFO:mascaf.validation:---- Ratio:             0.8755
INFO:mascaf.validation:---- Error:             -2.7782
INFO:mascaf.validation:---- Relative error:    -12.45%
INFO:mascaf.validation:Validation Results, accou

{'n_outside_nodes': 0, 'n_crossing_edges': 1, 'n_fully_outside_edges': 0}

## Terminal extension

Each branch tip is continued along its outward tangent by half a parent radius. The new tip radius is also half the parent radius.

In [51]:
morph.extend_terminals()
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera)
fig.show()

validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=12 sides=16 end_caps=False verts=384 faces=384
INFO:swctools.geometry:FrustaSet.from_swc_model edges=12 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=12 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 13 nodes, 12 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 5.5152
INFO:mascaf.validation:---- Ratio:             1.0799
INFO:mascaf.validation:---- Error:             0.4081
INFO:mascaf.validation:---- Relative error:    7.99%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   22.5125
INFO:mascaf.validation:---- Ratio:             1.0087
INFO:mascaf.validation:---- Error:             0.1949
INFO:mascaf.validation:---- Relative error:    0.87%
INFO:mascaf.validation:Validation Results, account_f

{'n_outside_nodes': 0, 'n_crossing_edges': 1, 'n_fully_outside_edges': 0}

## Surface-area normalization

Scale every radius by one factor so the cable surface area matches the mesh. Overlap between branches is left uncorrected.

In [56]:
morph.scale_radii_to_match_mesh(
    mm.mesh, metric="surface_area", account_for_overlaps=False
)
fig = visualize_mesh_cable_3d(mm, skeleton, morph, show_axes=False, camera=camera, show_nodes=True, node_size=8, centroid_line_width=5)
fig.show()


validator = Validation(mm, skeleton, morph)
validator.full_validation()

INFO:swctools.geometry:batch_frusta count=12 sides=16 end_caps=False verts=384 faces=384
INFO:swctools.geometry:FrustaSet.from_swc_model edges=12 sides=16 end_caps=False
INFO:swctools.viz:plot_model slider=False frusta=12 show_frusta=True show_centroid=True


INFO:mascaf.validation:Initialized Validation from MorphologyGraph
INFO:mascaf.validation:  Mesh: 1417 vertices, 2830 faces
INFO:mascaf.validation:  Skeleton: 137 nodes, 136 edges
INFO:mascaf.validation:  MorphologyGraph: 13 nodes, 12 edges
INFO:mascaf.validation:Validation Results, account_for_overlaps=False:
INFO:mascaf.validation:-- Volume Comparison:
INFO:mascaf.validation:---- Mesh volume:       5.1071
INFO:mascaf.validation:---- Morphology volume: 5.4341
INFO:mascaf.validation:---- Ratio:             1.0640
INFO:mascaf.validation:---- Error:             0.3270
INFO:mascaf.validation:---- Relative error:    6.40%
INFO:mascaf.validation:-- Surface Area Comparison:
INFO:mascaf.validation:---- Mesh area:         22.3177
INFO:mascaf.validation:---- Morphology area:   22.3177
INFO:mascaf.validation:---- Ratio:             1.0000
INFO:mascaf.validation:---- Error:             0.0000
INFO:mascaf.validation:---- Relative error:    0.00%
INFO:mascaf.validation:Validation Results, account_f

{'n_outside_nodes': 0, 'n_crossing_edges': 1, 'n_fully_outside_edges': 0}